# Tutorial 03: Finite Quantum Wells and Relativistic Klein Tunneling

> **Language / Idioma:** [Español](../es/03_pozos_cuanticos_y_efecto_klein.ipynb) | **English**

Using `ScalarPotential(func)`, you can define any custom potential profile $V(\mathbf{r})$ in Python and execute it inside the C++ FDTD integrator.

In this tutorial we demonstrate:
1. **Klein Tunneling:** A Dirac fermion hitting a supercritical potential step $V_0 > E + m$ exhibits non-zero transmission into the classically forbidden region (electron-positron pair-channel coupling).
2. **Finite Double Quantum Well:** Confinement and barrier tunneling between two adjacent wells.

In [ ]:
import sys
sys.path.append('../..')
import numpy as np
import matplotlib.pyplot as plt
from dirac_solver import (
    Grid, ConstantSpinor, GaussianPacket,
    DiracProblemBuilder, DiracSolver, ScalarPotential, Observables, _core
)

## 1. Klein Tunneling Across a Supercritical Potential Step ($V_0 > E + m$)

In [ ]:
grid = Grid(shape=(400,), spacing=(0.05,))
px, mass = 2.5, 1.0
E = np.sqrt(px**2 + mass**2)
V0 = E + mass + 1.5  # Supercritical Klein regime: V0 > E + m
print(f'Relativistic energy E = {E:.3f}, threshold E + m = {E + mass:.3f}, step V0 = {V0:.3f}')

spinor = ConstantSpinor(momentum=[px, 0.0, 0.0], mass=mass)
packet = GaussianPacket(spinor, center=[-4.0, 0.0, 0.0], spatial_width=0.8)

def klein_step(r):
    return V0 if r[0] >= 0.0 else 0.0

problem_klein = (
    DiracProblemBuilder()
    .set_grid(grid)
    .set_initial_state(packet)
    .set_potential(ScalarPotential(klein_step))
    .set_boundary_condition(_core.AbsorbingBoundary(strength=0.2, width=30))
    .set_time_parameters(time_step=0.001, total_time=5.5)
    .build()
)

solver_klein = DiracSolver(problem_klein)
obs_klein = Observables(solver_klein)
solver_klein.run_simulation(observables_handler=obs_klein)

x = grid.coords[:, 0]
rho = obs_klein.probability_density(solver_klein.get_psi())
dV = grid.get_cell_volume()
prob_left = np.sum(rho[x < 0]) * dV
prob_right = np.sum(rho[x >= 0]) * dV
T_coef = prob_right / (prob_left + prob_right)
print(f'Transmitted probability fraction (x >= 0): T = {T_coef:.4f}')

plt.figure(figsize=(10, 4))
plt.plot(x, rho, label='$\\rho(x, t_f)$')
plt.axvline(0.0, color='r', linestyle='--', label='Potential step $x=0$')
plt.xlabel('Position $x$')
plt.ylabel('Probability density $\\rho$')
plt.title(f'Klein Tunneling ($V_0 = {V0:.2f} > E + m$, $T \\approx {T_coef:.2f}$)')
plt.legend()
plt.grid(True)
plt.show()

## 2. Finite Double Quantum Well

In [ ]:
def double_well(r):
    x_val = r[0]
    if -4.0 <= x_val <= -0.5 or 0.5 <= x_val <= 4.0:
        return -1.5
    return 0.0

packet_dw = GaussianPacket(ConstantSpinor(momentum=[1.0, 0.0, 0.0]), center=[-2.2, 0.0, 0.0], spatial_width=0.6)
problem_dw = (
    DiracProblemBuilder()
    .set_grid(grid)
    .set_initial_state(packet_dw)
    .set_potential(ScalarPotential(double_well))
    .set_boundary_condition(_core.AbsorbingBoundary(strength=0.2, width=30))
    .set_time_parameters(time_step=0.001, total_time=4.0)
    .build()
)

solver_dw = DiracSolver(problem_dw)
solver_dw.run_simulation()
solver_dw.plot_probability_density()